# Module 3: Random Forest Classifier

step 1: re-load some of the variables used in previous modules:

promoter_1hot: (1000 x 800) feature matrix
y: ground-truth 1 / 0 expression classifier
X_train, X_test, y_train, y_test: matrix (promoter_1hot) and classifier train / test 80/20 splits
random_state = 42: seed for deterministic splits


In [1]:
# organize variables

import numpy as np
import pandas as pd

# load 1-hot encoded promoter file from file (2d array)
promoter_1hot = np.loadtxt("../data/promoters.1hot.csv", delimiter=",")
print(promoter_1hot.shape)

# load dataframe with metadata
df = pd.read_csv("../data/promoter_by_expression.csv")
print(df.head())

y = df['label'].values
print(f"X shape: {promoter_1hot.shape}")
print(f"y shape: {y.shape}")
print(f"Class balance: {y.mean():.2f} positive rate") # how many high expression / low expression promoters there are (?)

(1000, 800)
     gene_id                                           sequence  label  \
0  gene_0000  GTCGACGGATTCGCTATACGGTCCGATAAGTATAAAATAGTCGTCG...      1   
1  gene_0001  CTAAAAGATGTAGGGAACTATGCACATGAATATAAAGAACGATTCT...      1   
2  gene_0002  GACCACACGGACAAAGACGAAATGACAGCGTATAAAGGCAGAAGCT...      1   
3  gene_0003  GCGCAGTTGCCAGGTAAGTCTCTGATGCCATATAAACGGAAGCATG...      1   
4  gene_0004  AAGATCAACTACCTGAATAGCATACATTTGTATAAAAGTTACTGAC...      1   

   gc_content  tata_present  sequence_length  
0       0.375             1              200  
1       0.355             1              200  
2       0.380             1              200  
3       0.365             1              200  
4       0.345             1              200  
X shape: (1000, 800)
y shape: (1000,)
Class balance: 0.50 positive rate


In [7]:
# 3.2 Random forrest training skeleton

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split


rf = RandomForestClassifier(
    n_estimators=100,    # number of trees; start with 100
    max_features="sqrt",    # features considered per split; use 'sqrt'
    random_state=42,    # why does a random forest especially need this?
    n_jobs=-1            # use all CPU cores
)

# again split the train / test split. Same random state as in module 2, for consistency
X_train, X_test, y_train, y_test = train_test_split(
    promoter_1hot, y, test_size=0.2, random_state=42
)

# fit the rf model
rf.fit(X_train, y_train)

# Hard predictions and probability of class 1
rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1] # get only the second column (probability of being 1, I believe?)

# Evaluate
rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

# Training-set accuracy, to check for overfitting
rf_train_acc = rf.score(X_train, y_train)

print(f"RF train accuracy: {rf_train_acc:.3f}")
print(f"RF test accuracy:  {rf_acc:.3f}   (LR: 0.895)")
print(f"RF test AUC:       {rf_auc:.3f}   (LR: 0.957)")

RF train accuracy: 1.000
RF test accuracy:  0.935   (LR: 0.895)
RF test AUC:       0.982   (LR: 0.957)


In [8]:
# Again, do the cross-validation

from sklearn.model_selection import cross_val_score

rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=100, max_features='sqrt',
                           random_state=42, n_jobs=-1),
    promoter_1hot, y,              # full dataset -- matches what I did in Module 2
    cv=5, scoring='roc_auc'
)
print("RF AUC per fold:", rf_cv.round(3))
print(f"RF CV mean: {rf_cv.mean():.3f}, std: {rf_cv.std():.3f}   (LR: 0.952 ± 0.011)")

RF AUC per fold: [0.989 0.969 0.964 0.952 0.959]
RF CV mean: 0.967, std: 0.012   (LR: 0.952 ± 0.011)
